In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_environment", "dev", "environment")
v_env = dbutils.widgets.get("p_environment")

In [0]:
from pyspark.sql.functions import col, lit, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

In [0]:
movie_language_df = (
    spark.read
        .format("json")
        .option("multiline", "true")
        .load(f"{bronze_folder_path}/movie_language")
    )
display(movie_language_df)

languageId,languageRoleId,movieId
24574,2,36593
24574,2,36597
24574,2,36643
24574,2,36647
24574,2,36648
24574,2,36657
24574,2,36658
24574,2,36668
24574,2,36669
24574,2,36670


In [0]:
movie_language_schema = StructType([
    StructField("movieId", StringType()),
    StructField("languageId", IntegerType()),
    StructField("languageRoleId", IntegerType())
])

movie_language_df = (
    spark.read
        .schema(movie_language_schema)
        .format("json")
        .option("multiline", "true")
        .load(f"{bronze_folder_path}/movie_language")
)
display(movie_language_df)

movieId,languageId,languageRoleId
36593,24574,2
36597,24574,2
36643,24574,2
36647,24574,2
36648,24574,2
36657,24574,2
36658,24574,2
36668,24574,2
36669,24574,2
36670,24574,2


In [0]:
movie_language_df = (
    movie_language_df
        .withColumnsRenamed(
            {
                "movieId": "movie_Id",
                "languageId": "language_id"
            }

        )
        .withColumns(
            {
                "env": lit(v_env)
            }
        )
).drop("languageRoleId")

movie_language_df = add_ingestion_date(movie_language_df)

display(movie_language_df)

movie_Id,language_id,env,ingestion_date
36593,24574,dev,2026-10-05T06:12:17.301Z
36597,24574,dev,2026-10-05T06:12:17.301Z
36643,24574,dev,2026-10-05T06:12:17.301Z
36647,24574,dev,2026-10-05T06:12:17.301Z
36648,24574,dev,2026-10-05T06:12:17.301Z
36657,24574,dev,2026-10-05T06:12:17.301Z
36658,24574,dev,2026-10-05T06:12:17.301Z
36668,24574,dev,2026-10-05T06:12:17.301Z
36669,24574,dev,2026-10-05T06:12:17.301Z
36670,24574,dev,2026-10-05T06:12:17.301Z


In [0]:
(
    movie_language_df
        .write
        .format("parquet")
        .mode("overwrite")
        .save(f"{silver_folder_path}/movies_languages")
)

In [0]:
dbutils.notebook.exit("Success")